# 12. Bérleti Piac és Rent Gap Elemzés

**Cél:** a bérleti díjak fajlagos elemzése, a bruttó bérleti hozamok (yield) számítása és a befektetői tőkeallokáció vizsgálata.

---

### 📖 A bruttó bérleti hozam
A lakásvásárlás egyszerre lakhatás és tőkebefektetés; a befektetők kulcsmutatója a bruttó hozam (éves bérleti bevétel / vételár). A magasabb hozam jellemzően a kockázatosabb, alacsonyabb tőkeértékű piacokon érhető el; a prémium lokációkon a presztízsfelár alacsonyabb hozamot eredményez.

### 📖 Módszertani keret (bírálatra válaszul)
- **Hedonikusan PÁROSÍTOTT hozambecslés:** ugyanaz a regresszió fut az eladó és a kiadó mintán:
  ln(nm_ár) ~ log(korrigált alapterület) + szobaszám + állapotkód dummyk + panel + lift + erkély + vasúti immissziós sáv dummyk.
  A bruttó hozamot egy **50 m²-es, átlagos állapotú referencia-lakás PREDIKÁLT** eladói és bérleti nm-árából számítjuk (hozam = éves bérlet / eladási ár); a nyers medián-hányadost összehasonlításként kiírjuk.
- **Nettó hozam:** bruttó × kihasználtság (0.85) × (1 − költséghányad (0.15)).
- **Tőkésítési ráta:** 5% alapfeltevés 3.5%–6.5% érzékenységvizsgálattal — ez **feltevés, nem piaci kalibráció** (tranzakciós hozamadat nem áll rendelkezésre).
- **Térbeli rent gap:** a bruttó hozam és a rent gap (potenciális − aktuális tőkésített érték) immissziós sávonként (vasut_zona) és állapotonként is megjelenik.


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from ingatlan_tdk._utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np
import statsmodels.api as sm

df = load_szamitott_master()
display(HTML(render_area_header_html(df=df)))
elado = df[df['listing_type'] == 'elado'].copy()
kiado = df[df['listing_type'] == 'kiado'].copy()
print(f"Adatbázis: {len(elado)} db eladó és {len(kiado)} db kiadó hirdetés.")


### 1. Bérleti Díjak és Párosított Bruttó Hozam

**📌 Hedonikus párosítás:**
A kiadói és eladói nm-árakat ugyanazzal a specifikációval modellezzük, így a hozam egy azonos tulajdonságú referencia-lakás (50 m², 2 szoba, átlagos állapot — allapot_kod = 3, tégla, lift, erkély, 500-1000 m-es vasúti sáv) predikált értékeiből adódik:

bruttó hozam = (predikált havi bérleti nm-ár × 12) / predikált eladói nm-ár.

A modell csak a vasut_zona-val rendelkező (geokódolt) sorokon fut; a mintaelemszámot a kód kiírja. Összehasonlításként a párosítatlan nyers medián-hányados is megjelenik.


In [ ]:
# --- (a) Hedonikusan PÁROSÍTOTT hozambecslés ---
# UGYANAZ a regresszió fut az eladó és a kiadó mintán:
#   ln(nm_ar_huf) ~ log(korrigalt_alapterulet_nm) + szobaszam_osszes + allapot_kod dummyk
#                 + is_panel + has_lift + has_erkely + vasut_zona dummyk
# A bruttó hozamot a PREDIKÁLT értékekből számoljuk (50 m²-es, átlagos állapotú referencia-lakás).

REGS = ['nm_ar_huf', 'korrigalt_alapterulet_nm', 'szobaszam_osszes',
        'allapot_kod', 'is_panel', 'has_lift', 'has_erkely', 'vasut_zona']

def fit_hedonikus(minta):
    """Párosított hedonikus OLS: ln(nm-ár) a kanonikus magyarázóváltozókon (dummyk)."""
    d = minta.dropna(subset=REGS).copy()
    d = d[d['nm_ar_huf'] > 0].copy()
    X = pd.DataFrame({
        'log_meret': np.log(d['korrigalt_alapterulet_nm']),
        'szobaszam_osszes': d['szobaszam_osszes'].astype(float),
        'is_panel': d['is_panel'].astype(float),
        'has_lift': d['has_lift'].astype(float),
        'has_erkely': d['has_erkely'].astype(float),
    })
    X = X.loc[:, X.std() > 0]  # konstans oszlopok kihagyása
    ap = pd.get_dummies(d['allapot_kod'].astype(int), prefix='allapot').astype(float)
    z = pd.get_dummies(d['vasut_zona'].astype(str), prefix='zona').astype(float)
    ap_cats = sorted(d['allapot_kod'].astype(int).unique())
    zona_cats = [zz for zz in VASUT_IMMISSZIO_LABELS if (d['vasut_zona'].astype(str) == zz).any()]
    ap_base = f'allapot_{ap_cats[-1]}'   # referencia: legjobb megfigyelt állapot
    z_base = f'zona_{zona_cats[-1]}'     # referencia: legtávolabbi megfigyelt sáv
    X = pd.concat([X, ap.drop(columns=[ap_base]), z.drop(columns=[z_base])], axis=1)
    X = sm.add_constant(X)
    res = sm.OLS(np.log(d['nm_ar_huf']), X).fit()
    return res, d, ap_base, z_base, ap_cats, zona_cats

mod_elado, d_elado, ap_base_e, z_base_e, ap_cats_e, zona_cats_e = fit_hedonikus(elado)
mod_kiado, d_kiado, ap_base_k, z_base_k, ap_cats_k, zona_cats_k = fit_hedonikus(kiado)

print(f"Párosított modell minták (vasut_zona-val rendelkező, geokódolt sorok): "
      f"eladó n = {len(d_elado)}, kiadó n = {len(d_kiado)}")

def pred_ln(mod, ap_base, z_base, meret, szobak, panel, lift, erkely, kod, zona):
    """Referencia-profil predikciója: ln(nm-ár). Nem megfigyelt kategória esetén NaN."""
    p = mod.params
    v = p.get('const', 0.0)
    v += p.get('log_meret', 0.0) * np.log(meret)
    v += p.get('szobaszam_osszes', 0.0) * szobak
    v += p.get('is_panel', 0.0) * panel
    v += p.get('has_lift', 0.0) * lift
    v += p.get('has_erkely', 0.0) * erkely
    ak = f'allapot_{int(kod)}'
    if ak != ap_base:
        if ak in p.index:
            v += p[ak]
        else:
            return np.nan
    zk = f'zona_{zona}'
    if zk != z_base:
        if zk in p.index:
            v += p[zk]
        else:
            return np.nan
    return v

# Referencia-lakás: 50 m², 2 szoba, átlagos állapot (allapot_kod = 3), tégla, lift, erkély,
# 500-1000 m-es immissziós sáv (mindkét mintában megfigyelt kategória).
REF = dict(meret=50.0, szobak=2.0, panel=0.0, lift=1.0, erkely=1.0, kod=3, zona='500-1000 m')

ln_sell_50 = pred_ln(mod_elado, ap_base_e, z_base_e, **REF)
ln_rent_50 = pred_ln(mod_kiado, ap_base_k, z_base_k, **REF)
elado_nm_50 = float(np.exp(ln_sell_50))
berlet_nm_50 = float(np.exp(ln_rent_50))

brutto_hozam_parositott = berlet_nm_50 * 12 / elado_nm_50 * 100

# Nyers (párosítatlan) összehasonlítások
brutto_hozam_nyers_med = (kiado['nm_ar_huf'].median() * 12 / elado['nm_ar_huf'].median()) * 100
brutto_hozam_nyers_atlagmed = (kiado['nm_ar_huf'].mean() * 12 / elado['nm_ar_huf'].median()) * 100

print(f"Referencia-lakás: 50 m², 2 szoba, allapot_kod=3 (átlagos), tégla, lift, erkély, 500-1000 m sáv")
print(f"  -> predikált eladói nm-ár: {fmt_huf(elado_nm_50)}; predikált bérleti nm-ár: {fmt_huf(berlet_nm_50)}/hó")
print(f"  -> párosított bruttó hozam: {brutto_hozam_parositott:.2f}% "
      f"(nyers medián-hányados: {brutto_hozam_nyers_med:.2f}%)")

atlag_berlet_huf = kiado['price_huf'].mean()
median_berlet_huf = kiado['price_huf'].median()
atlag_berlet_nm = kiado['nm_ar_huf'].mean()

kpi_cards = [
    ("Átlagos Havi Bérlet", fmt_huf(atlag_berlet_huf) + " / hó", f"Medián: {fmt_huf(median_berlet_huf)}", "#1e3a8a"),
    ("Bérleti Fajlagos Díj", fmt_huf(atlag_berlet_nm) + " / m²", "Havi fajlagos díj", "#2563eb"),
    ("Párosított Bruttó Hozam (50 m² ref.)", f"{brutto_hozam_parositott:.2f}%", "Hedonikus modell predikcióiból", "#059669"),
    ("Nyers Medián-Hányados", f"{brutto_hozam_nyers_med:.2f}%", "Párosítatlan összehasonlítás", "#d97706"),
    ("Kiadó Lakások Aránya", f"{len(kiado)/len(df)*100:.1f}%", f"{len(kiado)} db hirdetés", "#7c3aed"),
]
display(HTML(kpi_grid_html(kpi_cards)))

# Hozambecslések összehasonlító táblázata (párosított vs nyers)
nyers_df = pd.DataFrame({
    'Becslés': [
        'Párosított hedonikus (50 m² referencia-lakás)',
        'Nyers: medián kiadói nm-ár × 12 / medián eladói nm-ár',
        'Nyers (eredeti): átlag kiadói nm-ár × 12 / medián eladói nm-ár',
    ],
    'Bruttó hozam': [
        f'{brutto_hozam_parositott:.2f}%',
        f'{brutto_hozam_nyers_med:.2f}%',
        f'{brutto_hozam_nyers_atlagmed:.2f}%',
    ],
})
display(HTML("<div style='max-width: 760px; margin: 15px 0;'>" + nyers_df.to_html(classes='table table-bordered table-striped', index=False) + "</div>"))

# A két modell együtthatói egymás mellett (párosított becslés átláthatósága)
coef_df = pd.DataFrame({'eladó': mod_elado.params, 'kiadó': mod_kiado.params}).round(3)
coef_df.columns = ['Eladó: ln(nm-ár) együttható', 'Kiadó: ln(nm-ár) együttható']
coef_df = coef_df.reset_index().rename(columns={'index': 'Változó'})
display(HTML("<div style='max-width: 760px; margin: 15px 0;'>" + coef_df.to_html(classes='table table-bordered table-striped', index=False) + "</div>"))


### 2. Nettó Hozam és Tőkésítési Ráta Érzékenységvizsgálat

**📌 Nettó hozam képlete:**

nettó hozam = bruttó hozam × kihasználtság (0.85) × (1 − költséghányad (0.15))

**📌 Tőkésítési ráta (cap rate):**
A tőkésítéshez használt 5% **feltevés — nem piaci kalibráció**, mert tranzakciós hozamadat nem áll rendelkezésre. Az érzékenységvizsgálat ezért 3.5%, 5% és 6.5% tőkésítési rátával fut.


In [ ]:
# --- (b) Nettó hozam: bruttó × kihasználtság × (1 − költséghányad) ---
KIHASZNALTSAG = 0.85
KOLTSEGHANYAD = 0.15
netto_hozam = brutto_hozam_parositott * KIHASZNALTSAG * (1 - KOLTSEGHANYAD)

# --- (c) Érzékenységvizsgálat: 3.5% / 5% / 6.5% tőkésítési ráta (FELTEVÉS, nem piaci kalibráció) ---
cap_rates = [0.035, 0.05, 0.065]
tokesitett_50m2_5pct = berlet_nm_50 * 50 * 12 / 0.05
sens = pd.DataFrame({
    'Tőkésítési ráta': [f"{r*100:.1f}%" for r in cap_rates],
    'Tőkésített érték (50 m² ref. lakás)': [berlet_nm_50 * 50 * 12 / r for r in cap_rates],
})
sens['Értékváltozás az 5%-os alaphoz képest'] = (sens['Tőkésített érték (50 m² ref. lakás)'] / tokesitett_50m2_5pct - 1) * 100

kpi_cards = [
    ("Nettó Hozam", f"{netto_hozam:.2f}%",
     f"Bruttó {brutto_hozam_parositott:.2f}% × {KIHASZNALTSAG:.2f} kihaszn. × (1 − {KOLTSEGHANYAD:.2f} költségh.)", "#10b981"),
    ("Tőkésített Érték @ 5%", fmt_huf(tokesitett_50m2_5pct),
     "50 m² referencia-lakás éves bérlete / 0.05", "#059669"),
]
display(HTML(kpi_grid_html(kpi_cards)))

sens_disp = sens.copy()
sens_disp['Tőkésített érték (50 m² ref. lakás)'] = sens_disp['Tőkésített érték (50 m² ref. lakás)'].map(fmt_huf)
sens_disp['Értékváltozás az 5%-os alaphoz képest'] = sens_disp['Értékváltozás az 5%-os alaphoz képest'].map(lambda v: f'{v:+.1f}%')
display(HTML("<div style='max-width: 760px; margin: 15px 0;'>" + sens_disp.to_html(classes='table table-bordered table-striped', index=False) + "</div>"))

fig_sens = px.bar(
    sens, x='Tőkésítési ráta', y='Tőkésített érték (50 m² ref. lakás)',
    title='Tőkésített érték érzékenysége a tőkésítési rátára (50 m² referencia-lakás; 5% feltevés, nem piaci kalibráció)',
    labels={'Tőkésített érték (50 m² ref. lakás)': 'Tőkésített érték (HUF)'},
    template=PLOTLY_TEMPLATE, text_auto='.3s'
)
fig_sens.update_layout(height=400)
fig_sens.show()


### 3. Térbeli Rent Gap — Immissziós Sávok × Állapot

**📌 Definíciók (a fenti párosított becslésből):**
- **Aktuális tőkésített bérleti érték:** az adott sávban és állapotban a referencia-profilra predikált havi bérleti nm-ár × 12 / tőkésítési ráta (5%, feltevés).
- **Potenciális tőkésített érték:** ugyanabban a sávban a legmagasabb predikált bérleti nm-árú állapotkategória (highest-and-best-use feltevés) tőkésítve.
- **Rent gap = potenciális − aktuális tőkésített érték (Ft/m²).**
- A sávonkénti **bruttó hozam** az eladó és a kiadó modell párosított predikcióiból számítódik.

A grouped bar ábra a rent gap-et mutatja immissziós sávonként (x tengely) és állapotonként (sávcsoport).


In [ ]:
# --- (d) Térbeli rent gap: immissziós sávonként × állapotonként ---
CAP = 0.05  # tőkésítési ráta — feltevés (lásd a 2. szakasz érzékenységvizsgálatát)

allapot_nev = {int(k): str(v) for k, v in df.groupby('allapot_kod')['allapot'].first().items()}
zonak = [zz for zz in VASUT_IMMISSZIO_LABELS if zz in zona_cats_e and zz in zona_cats_k]
kods = sorted(set(ap_cats_e) & set(ap_cats_k))

def rent_pred(k, zz):
    return float(np.exp(pred_ln(mod_kiado, ap_base_k, z_base_k, REF['meret'], REF['szobak'],
                                REF['panel'], REF['lift'], REF['erkely'], k, zz)))

def sell_pred(k, zz):
    return float(np.exp(pred_ln(mod_elado, ap_base_e, z_base_e, REF['meret'], REF['szobak'],
                                REF['panel'], REF['lift'], REF['erkely'], k, zz)))

# Potenciális = az adott sávban a legmagasabb predikált bérleti nm-árú állapot (highest-and-best-use)
pot_by_zona = {zz: max(((rent_pred(k, zz), k) for k in kods), key=lambda t: t[0]) for zz in zonak}
print('Potenciális (legmagasabb predikált bérleti nm-árú) állapot sávonként:',
      {zz: allapot_nev.get(kod, kod) for zz, (_, kod) in pot_by_zona.items()})

rows = []
for zz in zonak:
    pot_nm, _ = pot_by_zona[zz]
    for k in kods:
        rent_nm = rent_pred(k, zz)
        sell_nm = sell_pred(k, zz)
        rows.append({
            'vasut_zona': zz,
            'allapot': allapot_nev.get(k, k),
            'allapot_kod': k,
            'brutto_hozam_pct': rent_nm * 12 / sell_nm * 100,
            'rent_gap': (pot_nm - rent_nm) * 12 / CAP,
        })
gap_df = pd.DataFrame(rows)

legnagyobb = gap_df.loc[gap_df['rent_gap'].idxmax()]
print(f"Legnagyobb becsült rent gap: {fmt_huf(legnagyobb['rent_gap'])}/m² — "
      f"sáv: {legnagyobb['vasut_zona']}, állapot: {legnagyobb['allapot']}")

# 1) Bruttó hozam sávonként (referencia-állapot, allapot_kod = 3)
yield_zona = gap_df[gap_df['allapot_kod'] == 3][['vasut_zona', 'brutto_hozam_pct']].copy()
fig_yield = px.bar(
    yield_zona, x='vasut_zona', y='brutto_hozam_pct',
    title='Bruttó bérleti hozam immissziós sávonként (párosított hedonikus becslés, 50 m² ref. lakás, átlagos állapot)',
    labels={'vasut_zona': 'Vasúti immissziós sáv', 'brutto_hozam_pct': 'Bruttó hozam (%)'},
    category_orders={'vasut_zona': zonak}, template=PLOTLY_TEMPLATE, text_auto='.2f'
)
fig_yield.update_layout(height=400)
fig_yield.show()

# 2) Rent gap grouped bar: sáv × állapot
szinek = ['#f87171', '#fb923c', '#facc15', '#4ade80', '#2dd4bf', '#60a5fa', '#a78bfa']
fig_gap = go.Figure()
for i, k in enumerate(kods):
    sub = gap_df[gap_df['allapot_kod'] == k]
    fig_gap.add_trace(go.Bar(
        x=sub['vasut_zona'], y=sub['rent_gap'],
        name=str(allapot_nev.get(k, k)),
        marker_color=szinek[i % len(szinek)]
    ))
fig_gap.update_layout(
    barmode='group',
    title='Térbeli Rent Gap: potenciális − aktuális tőkésített bérleti érték (Ft/m²) sávonként és állapotonként (5% feltevés)',
    xaxis=dict(categoryorder='array', categoryarray=zonak, tickangle=-20),
    xaxis_title='Vasúti immissziós sáv',
    yaxis_title='Rent gap (Ft/m², tőkésítve 5%-kal)',
    template=PLOTLY_TEMPLATE, height=470, legend_title_text='Állapot'
)
fig_gap.show()

# 3) Rent gap mátrix táblázat
mat = gap_df.pivot(index='vasut_zona', columns='allapot', values='rent_gap').reindex(zonak).round(0)
display(HTML("<div style='max-width: 860px; margin: 15px 0;'>" + mat.reset_index().rename(columns={'vasut_zona': 'Immissziós sáv'}).to_html(classes='table table-bordered table-striped', index=False, na_rep='n/a') + "</div>"))


### 4. Bérleti Díjak és Lakásméret Összefüggése

**📌 Mit mutat a scatter pontdiagram?**
A havi bérleti díj alakulását a lakásméret függvényében. A trendvonal (OLS) és a cellában kiírt predikált díjak (35 m² és 60 m² referencia-méretekre) a kód által számított értékek; a szöveges állítások ezekre épülnek.


In [ ]:
# Kiadó lakások méret vs bérleti díj szórásdiagramja
fig3 = px.scatter(
    kiado,
    x='alapterulet_nm',
    y='price_huf',
    color='varosresz',
    trendline='ols',
    title='Alapterület vs. Havi bérleti díj a kiadó lakások piacán',
    labels={'alapterulet_nm': 'Alapterület (m²)', 'price_huf': 'Bérleti díj (HUF / hó)'},
    template=PLOTLY_TEMPLATE
)
fig3.update_layout(height=450)
fig3.show()

# Dinamikus (kód által számított) hivatkozási értékek — a markdown-állítások ezekre épülnek
m_ok = kiado.dropna(subset=['alapterulet_nm', 'price_huf'])
m_ok = m_ok[(m_ok['price_huf'] > 0) & (m_ok['alapterulet_nm'] > 0)]
x = m_ok['alapterulet_nm'].to_numpy(dtype=float)
y = m_ok['price_huf'].to_numpy(dtype=float)
meredekseg = float(np.cov(x, y)[0, 1] / np.var(x))
tengelymetszet = float(np.mean(y) - meredekseg * np.mean(x))
pred_35 = tengelymetszet + meredekseg * 35
pred_60 = tengelymetszet + meredekseg * 60
print(f"OLS trend (n = {len(m_ok)}): havi bérleti díj = {tengelymetszet:,.0f} Ft + {meredekseg:,.0f} Ft/m² × alapterület")
print(f"  -> 35 m²-re predikált díj: {fmt_huf(pred_35)}/hó; 60 m²-re predikált díj: {fmt_huf(pred_60)}/hó")


### 5. Interaktív Bérleti Hozam és Rent Gap Kalkulátor

**📌 Számolja ki saját befektetésének megtérülését!**
Állítsa be a vételárat, a lakás területét, a várt bérleti díjat, a kihasználtságot, a költséghányadot és a **tőkésítési rátát (cap rate)** — a cap rate **felhasználói input** (3.5%–6.5% csúszka), nem a saját hozamból számítódik.

**📌 Realizált rent gap képlete (kategóriákat nem keverünk):**

tőkésített rent gap = 12 × (javított − jelenlegi havi nm-bér) × terület / cap rate

A felújítási költség NINCS levonva a gap-ből; azt a külön „Beruházás feletti nettó többlet” kártya mutatja (tőkésített gap − felújítási költség).


In [ ]:
# Valós idejű Neil Smith Rent Gap és Bérleti Megtérülés Kalkulátor
# (e) cap_rate = FELHASZNÁLÓI INPUT (csúszka), nem a saját hozamból származik;
#     a realizált rent gap képlete: 12 x (javított - jelenlegi havi nm-bér) x terület / cap_rate
html_rent_gap = """
<div id="rent_gap_app" style="background:#ffffff; border:1px solid #cbd5e1; border-radius:12px; padding:22px; box-shadow:0 4px 6px -1px rgba(0,0,0,0.06); margin:18px 0; font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Helvetica,Arial,sans-serif;">
  <div style="display:flex; justify-content:space-between; align-items:center; border-bottom:2px solid #f1f5f9; padding-bottom:12px; margin-bottom:18px;">
    <div>
      <h3 style="margin:0; color:#1e3a8a; font-size:19px; font-weight:700;">📊 Neil Smith Rent Gap & Bérleti Megtérülés Kalkulátor</h3>
      <p style="margin:3px 0 0 0; color:#64748b; font-size:13px;">Dinamikus hozamszámítás, felújítási értéknövekmény és járadék-rés realizáció valós időben</p>
    </div>
    <span style="background:#dbeafe; color:#1d4ed8; font-size:11px; font-weight:700; padding:4px 10px; border-radius:9999px;">Kliensoldali JS Motor</span>
  </div>

  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(280px, 1fr)); gap:20px; margin-bottom:20px;">
    <!-- 1. oszlop: Bázis paraméterek -->
    <div style="background:#f8fafc; padding:16px; border-radius:8px; border:1px solid #e2e8f0;">
      <div style="font-weight:700; color:#334155; margin-bottom:12px; font-size:14px; text-transform:uppercase; letter-spacing:0.5px;">1. Bázis Ingatlan és Bérleti Díj</div>

      <div style="margin-bottom:12px;">
        <div style="display:flex; justify-content:space-between; font-size:13px; font-weight:600; color:#475569;">
          <span>Vételár (M Ft):</span> <span id="rg_lbl_vetel" style="color:#2563eb; font-weight:700;">50 M Ft</span>
        </div>
        <input type="range" id="rg_vetel" min="30" max="120" value="50" step="5" style="width:100%; accent-color:#2563eb;" oninput="recalcRentGap()">
      </div>

      <div style="margin-bottom:12px;">
        <div style="display:flex; justify-content:space-between; font-size:13px; font-weight:600; color:#475569;">
          <span>Havi bérleti díj (ezer Ft):</span> <span id="rg_lbl_berlet" style="color:#2563eb; font-weight:700;">250 ezer Ft/hó</span>
        </div>
        <input type="range" id="rg_berlet" min="150" max="500" value="250" step="10" style="width:100%; accent-color:#2563eb;" oninput="recalcRentGap()">
      </div>

      <div style="margin-bottom:12px;">
        <div style="display:flex; justify-content:space-between; font-size:13px; font-weight:600; color:#475569;">
          <span>Lakás területe:</span> <span id="rg_lbl_terulet" style="color:#2563eb; font-weight:700;">50 m²</span>
        </div>
        <input type="range" id="rg_terulet" min="20" max="120" value="50" step="5" style="width:100%; accent-color:#2563eb;" oninput="recalcRentGap()">
      </div>

      <div style="margin-bottom:12px;">
        <div style="display:flex; justify-content:space-between; font-size:13px; font-weight:600; color:#475569;">
          <span>Kihasználtsági ráta:</span> <span id="rg_lbl_occ" style="color:#059669; font-weight:700;">95%</span>
        </div>
        <input type="range" id="rg_occ" min="70" max="100" value="95" step="5" style="width:100%; accent-color:#059669;" oninput="recalcRentGap()">
      </div>

      <div style="margin-bottom:12px;">
        <div style="display:flex; justify-content:space-between; font-size:13px; font-weight:600; color:#475569;">
          <span>Üzemeltetés & amortizáció:</span> <span id="rg_lbl_cost" style="color:#dc2626; font-weight:700;">15%</span>
        </div>
        <input type="range" id="rg_cost" min="5" max="30" value="15" step="5" style="width:100%; accent-color:#dc2626;" oninput="recalcRentGap()">
      </div>

      <div style="margin-bottom:6px;">
        <div style="display:flex; justify-content:space-between; font-size:13px; font-weight:600; color:#475569;">
          <span>Tőkésítési ráta (cap rate):</span> <span id="rg_lbl_cap" style="color:#7c3aed; font-weight:700;">5.00%</span>
        </div>
        <input type="range" id="rg_cap" min="3.5" max="6.5" value="5" step="0.25" style="width:100%; accent-color:#7c3aed;" oninput="recalcRentGap()">
      </div>
    </div>

    <!-- 2. oszlop: Felújítás és Rent Gap realizáció -->
    <div style="background:#f8fafc; padding:16px; border-radius:8px; border:1px solid #e2e8f0;">
      <div style="font-weight:700; color:#334155; margin-bottom:12px; font-size:14px; text-transform:uppercase; letter-spacing:0.5px;">2. Rent Gap Értéknövelő Beruházás</div>

      <div style="margin-bottom:14px;">
        <div style="display:flex; justify-content:space-between; font-size:13px; font-weight:600; color:#475569;">
          <span>Felújítási tőkeráfordítás (M Ft):</span> <span id="rg_lbl_felujitas" style="color:#7c3aed; font-weight:700;">5.0 M Ft</span>
        </div>
        <input type="range" id="rg_felujitas" min="0" max="25" value="5" step="1" style="width:100%; accent-color:#7c3aed;" oninput="recalcRentGap()">
      </div>

      <div style="margin-bottom:14px;">
        <div style="display:flex; justify-content:space-between; font-size:13px; font-weight:600; color:#475569;">
          <span>Bérletnövekedés felújítás után:</span> <span id="rg_lbl_novek" style="color:#059669; font-weight:700;">+30%</span>
        </div>
        <input type="range" id="rg_novek" min="0" max="60" value="30" step="5" style="width:100%; accent-color:#059669;" oninput="recalcRentGap()">
      </div>

      <div style="background:#eff6ff; border-left:4px solid #2563eb; padding:10px 12px; border-radius:4px; font-size:12px; color:#1e40af; margin-top:10px;">
        🏢 <b>Neil Smith tézis:</b> A járadék-rés (Rent Gap) a felújítás nélküli aktuális tőkésített bérleti érték és a legmagasabb minőségű (potenciális) tőkésített érték közötti különbség. A cap rate felhasználói beállítás (3.5–6.5%), nem a saját hozamból számítódik.
      </div>
    </div>
  </div>

  <!-- KPI Kártyák -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:14px; margin-bottom:10px;">
    <div style="background:#eff6ff; border:1px solid #93c5fd; border-radius:10px; padding:14px 18px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#1e40af; text-transform:uppercase;">Bruttó Bérleti Hozam</div>
      <div id="rg_res_brutto" style="font-size:26px; font-weight:800; color:#1d4ed8; margin:4px 0;">--%</div>
      <div style="font-size:11px; color:#2563eb;">Kínálati vételárra vetítve</div>
    </div>

    <div style="background:#f0fdf4; border:1px solid #86efac; border-radius:10px; padding:14px 18px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#166534; text-transform:uppercase;">Tiszta Nettó Hozam</div>
      <div id="rg_res_netto" style="font-size:26px; font-weight:800; color:#15803d; margin:4px 0;">--%</div>
      <div style="font-size:11px; color:#16a34a;">Üresedés & fenntartás után</div>
    </div>

    <div style="background:#fffbeb; border:1px solid #fde68a; border-radius:10px; padding:14px 18px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#92400e; text-transform:uppercase;">Valós Megtérülés</div>
      <div id="rg_res_payback" style="font-size:24px; font-weight:800; color:#b45309; margin:4px 0;">-- év</div>
      <div style="font-size:11px; color:#d97706;">Nettó cash flow alapján</div>
    </div>

    <div style="background:#faf5ff; border:1px solid #d8b4fe; border-radius:10px; padding:14px 18px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#6b21a8; text-transform:uppercase;">Realizálható Rent Gap</div>
      <div id="rg_res_gap" style="font-size:24px; font-weight:800; color:#7e22ce; margin:4px 0;">-- M Ft</div>
      <div style="font-size:11px; color:#9333ea;">12 × (javított − jelenlegi nm-bér) × terület / cap rate</div>
    </div>

    <div style="background:#fef2f2; border:1px solid #fecaca; border-radius:10px; padding:14px 18px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#991b1b; text-transform:uppercase;">Beruházás Feletti Nettó Többlet</div>
      <div id="rg_res_netgap" style="font-size:24px; font-weight:800; color:#b91c1c; margin:4px 0;">-- M Ft</div>
      <div style="font-size:11px; color:#dc2626;">Tőkésített gap − felújítási költség (külön mutató)</div>
    </div>
  </div>
</div>

<script>
(function() {
  function update() {
    const vetel = parseFloat(document.getElementById('rg_vetel').value) * 1e6;
    const berlet = parseFloat(document.getElementById('rg_berlet').value) * 1e3;
    const terulet = parseFloat(document.getElementById('rg_terulet').value);
    const occ = parseFloat(document.getElementById('rg_occ').value) / 100;
    const cost = parseFloat(document.getElementById('rg_cost').value) / 100;
    const cap = parseFloat(document.getElementById('rg_cap').value) / 100;
    const felujitas = parseFloat(document.getElementById('rg_felujitas').value) * 1e6;
    const novek = parseFloat(document.getElementById('rg_novek').value) / 100;

    document.getElementById('rg_lbl_vetel').innerText = (vetel / 1e6).toFixed(0) + ' M Ft';
    document.getElementById('rg_lbl_berlet').innerText = (berlet / 1e3).toFixed(0) + ' ezer Ft/hó';
    document.getElementById('rg_lbl_terulet').innerText = terulet.toFixed(0) + ' m²';
    document.getElementById('rg_lbl_occ').innerText = Math.round(occ * 100) + '%';
    document.getElementById('rg_lbl_cost').innerText = Math.round(cost * 100) + '%';
    document.getElementById('rg_lbl_cap').innerText = (cap * 100).toFixed(2) + '%';
    document.getElementById('rg_lbl_felujitas').innerText = (felujitas / 1e6).toFixed(1) + ' M Ft';
    document.getElementById('rg_lbl_novek').innerText = '+' + Math.round(novek * 100) + '%';

    const eves_brutto = berlet * 12;
    const brutto_h = (eves_brutto / vetel) * 100;
    const netto_eves = (eves_brutto * occ) * (1.0 - cost);
    const netto_h = (netto_eves / vetel) * 100;
    const megterules = netto_eves > 0 ? vetel / netto_eves : 0;

    // Rent gap — kategóriákat NEM keverünk (csak bérleti nm-árak és terület):
    // tőkésített gap = 12 x (javított - jelenlegi havi nm-bér) x terület / cap_rate
    const jelenlegi_nm = berlet / terulet;
    const javitott_nm = jelenlegi_nm * (1.0 + novek);
    const eves_tobblet = 12.0 * (javitott_nm - jelenlegi_nm) * terulet;
    const tokesitett_gap = cap > 0 ? eves_tobblet / cap : 0;
    const netto_tobblet = tokesitett_gap - felujitas;

    document.getElementById('rg_res_brutto').innerText = brutto_h.toFixed(2) + '%';
    document.getElementById('rg_res_netto').innerText = netto_h.toFixed(2) + '%';
    document.getElementById('rg_res_payback').innerText = megterules.toFixed(1) + ' év';
    document.getElementById('rg_res_gap').innerText = (tokesitett_gap > 0 ? '+' : '') + (tokesitett_gap / 1e6).toFixed(1) + ' M Ft';
    document.getElementById('rg_res_netgap').innerText = (netto_tobblet > 0 ? '+' : '') + (netto_tobblet / 1e6).toFixed(1) + ' M Ft';
  }

  window.recalcRentGap = update;
  setTimeout(update, 50);
})();
</script>
"""
display(HTML(html_rent_gap))
